In [1]:
# Standard imports
import polars as pl
import numpy as np
from pathlib import Path

# Scikit-learn (baseline model and preprocessing)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.metrics import log_loss, roc_auc_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
# Matplotlib/Seaborn styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,5)
plt.rcParams["figure.dpi"] = 100

# Configuration
from IPython.display import display

# Polars display settings
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(15)

# Paths to data
TRAIN_PATH = Path("../data/interim/train_split.parquet")
VAL_PATH = Path("../data/interim/val_split.parquet")

# Verify files exist
print(f"Polars version: {pl.__version__}")
print(f"Train file exists: {TRAIN_PATH.exists()}")
print(f"Validation file exists: {VAL_PATH.exists()}")

Polars version: 1.39.3
Train file exists: True
Validation file exists: True


## Feature engineering

### Time features

In [2]:
# Load train and val lazy frames
train_lazy = pl.scan_parquet(TRAIN_PATH)
val_lazy = pl.scan_parquet(VAL_PATH)

# Function to add time features
def add_time_features(df):
    """Extract hour_of_day, day_of_week, and is_weekend from 'hour' column.

    Avazu hour format: YYMMDDHH (e.g., 14102100 = 2014-10-21 00:00).
    """
    return (
        df
        .with_columns(
            pl.col("hour").cast(pl.String).alias("hour_str")
        )
        .with_columns([
            pl.col("hour_str").str.slice(4,2).cast(pl.Int8).alias("day"),
            pl.col("hour_str").str.slice(6,2).cast(pl.Int8).alias("hour_of_day")
        ])
        .with_columns(
            # 21.10.2014 was a Tuesday (since Monday=0) --> day_of_week = 1
            # Day 21 = weekday 1, ..., day 27 = 0 (Monday)
            ((pl.col("day") - 21 + 1) % 7).alias("day_of_week")
        )
        .with_columns(
            # for weekend: Saturday=5, Sunday=6
            pl.col("day_of_week").is_in([5,6]).alias("is_weekend")
        )
        .drop("hour_str") # this columns is not needed anymore
    )

# Apply to both splits
train_lazy = add_time_features(train_lazy)
val_lazy = add_time_features(val_lazy)

# Quick check on train (show first few rows of new features) 
check = (
    train_lazy
    .select(["day","day_of_week","is_weekend"])
    # Sampling different days to verify time features correctly
    .unique()
    .sort("day")
    .collect()
)

# And check on val
check_val = (
    val_lazy
    .select(["day", "day_of_week", "is_weekend"])
    .unique()
    .sort("day")
    .collect()
)

display(check)
display(check_val)

day,day_of_week,is_weekend
i8,i8,bool
21,1,false
22,2,false
23,3,false
24,4,false
25,5,true
26,6,true
27,0,false
28,1,false


day,day_of_week,is_weekend
i8,i8,bool
29,2,false
30,3,false


## is_app flag (traffic type)

In [3]:
# Placeholder values from EDA
SITE_PLACEHOLDER_IN_APP_ID = "ecad2386"

# Function to add is_app flag
def add_is_app_flag(df):
    """Add is_app flag (boolean) based on app_id placeholder.
    
    If app_id equals the site placeholder, it's site traffic (False). Otherwise - app traffic (True).
    """
    return (
        df
        .with_columns(
            (pl.col("app_id") != SITE_PLACEHOLDER_IN_APP_ID).alias("is_app")
        )
    )

# Apply to both splits
train_lazy = add_is_app_flag(train_lazy)
val_lazy = add_is_app_flag(val_lazy)

# Quick check (CTR per is_app in train set)
check = (
    train_lazy
    .group_by("is_app")
    .agg([
        pl.len().alias("impressions"),
        pl.col("click").sum().alias("clicks")
    ])
    .with_columns(
        (pl.col("clicks") / pl.col("impressions")).alias("ctr")
    )
    .sort("is_app")
    .collect()
)

display(check)

is_app,impressions,clicks,ctr
bool,u32,i64,f64
false,21135275,4233200,0.200291
true,11242146,1317640,0.117205


In [4]:
# See current columns in train
print(f"Current columns in train:\n {train_lazy.collect_schema().names()}")


Current columns in train:
 ['id', 'click', 'hour', 'C1', 'banner_pos', 'site_id', 'site_domain', 'site_category', 'app_id', 'app_domain', 'app_category', 'device_id', 'device_ip', 'device_model', 'device_type', 'device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'day', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_app']


In [5]:
# Sanity chech: distribution of new time features
sanity_check = (
    train_lazy
    .select([
        pl.col("day").min().alias("day_min"),
        pl.col("day").max().alias("day_max"),
        pl.col("hour_of_day").min().alias("hour_of_day_min"),
        pl.col("hour_of_day").max().alias("hour_of_day_max"),
        pl.col("day_of_week").min().alias("day_of_week_min"),
        pl.col("day_of_week").max().alias("day_of_week_max"),
        pl.col("is_weekend").sum().alias("weekend_rows"),
        pl.col("is_app").sum().alias("app_rows"),
        pl.len().alias("total"),
    ])
    .collect()
)

display(sanity_check)

day_min,day_max,hour_of_day_min,hour_of_day_max,day_of_week_min,day_of_week_max,weekend_rows,app_rows,total
i8,i8,i8,i8,i8,i8,u32,u32,u32
21,28,0,23,0,6,7199014,11242146,32377421


In [6]:
# Columns to drop (not useful for modeling)
DROP_COLS = ["id","day","hour"]

# Prepare features and target
def prepare_xy(df_lazy):
    """ Separate features from target, drop useless columns."""
    df = df_lazy.drop(DROP_COLS).collect()
    y = df["click"].to_numpy() # target (scikit-learn wants numpy arrays)
    X = df.drop("click") # all features (not converted to numpy array yet, first - encoding)
    return X, y

X_train, y_train = prepare_xy(train_lazy)
X_val, y_val = prepare_xy(val_lazy)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"\nTrain CTR (check): {y_train.mean():.4f}")
print(f"Val CTR (check): {y_val.mean():.4f}")    

X_train shape: (32377421, 25)
y_train shape: (32377421,)
X_val shape: (8051546, 25)
y_val shape: (8051546,)

Train CTR (check): 0.1714
Val CTR (check): 0.1632


## Feature encoding

In [7]:
# Columns grouped by encoding strategy

# One-hot encoding - low cardinality (all categorical features with <100 unique values)
ONEHOT_COLS = [
    # Low cardinality (<100 unique values)
    "C1","banner_pos","device_type","device_conn_type",
    "C15","C16","C18",
    "app_category","site_category",
    "hour_of_day","day_of_week",
    # Boolean feature (treat as categorical)
    "is_weekend", "is_app",
]

# Feature hashing - medium and high cardinality
HASH_COLS = [
    # Medium cardinality (100-10000 unique values)
    "C14", "C17","C19","C20","C21",
    "app_domain","app_id","site_domain","site_id","device_model",
    # very high cardinality values
    "device_id", "device_ip"]

# Bucket counts per column - matched to cardinality
# Lower cardinality -> fewer buckets (avoid sparsity)  
# Higher cardinality -> more buckets (avoid collisions)
HASH_BUCKETS = { # creating a dictionary of column names (key) and numbers of buckets for each (values) 
    # Medium cardinality (fewer buckets)
    "C14": 1000, # ~2600 unique
    "C17": 500, # ~435
    "C19": 100, # ~68
    "C20": 200, # ~172
    "C21": 100, # ~60
    "app_domain": 500, # ~559
    # Higher cardinality (more buckets)
    "app_id": 2000, # ~8500
    "site_id": 2000, # ~4700
    "site_domain": 2000, # ~7700
    "device_model": 2000, # ~8200
    # Mega-high cardinality
    "device_id": 5000, # 2.7M
    "device_ip": 5000, # 6.7M
}

# Verify all feature columns are accounted for
all_feature_cols = set(ONEHOT_COLS) | set(HASH_COLS)
actual_cols = set(X_train.columns)
missing = actual_cols - all_feature_cols # should be empty
extra = all_feature_cols - actual_cols # should be empty

print(f"Total one-hot columns: {len(ONEHOT_COLS)}")
print(f"Total hash columns: {len(HASH_COLS)}")
print(f"Total expected: {len(ONEHOT_COLS)+len(HASH_COLS)}")
print(f"Actual X columns: {len(actual_cols)}\n")
print(f"Missing from strategy: {missing}")
print(f"Extra in strategy: {extra}")

total_hash_cols = sum(HASH_BUCKETS.values())
print(f"\nTotal hash output columns: {total_hash_cols:,}")

Total one-hot columns: 13
Total hash columns: 12
Total expected: 25
Actual X columns: 25

Missing from strategy: set()
Extra in strategy: set()

Total hash output columns: 20,400


### Feature hashing

In [8]:
# Feature hashing for medium and high cardinality columns
from scipy.sparse import hstack, vstack # stack sparse matrices horizontally

def hash_columns(df, cols_with_buckets):
    """ Hash multiple columns to different number of buckets using FeatureHasher.

    Returns single scipy sparse matrix with all hashed columns stacked horizontally (shape (n_rows, n_buckets)).
    """
    hashed_parts = []
    
    for col, n_buckets in cols_with_buckets.items():
        # FeatureHasher expects iterable of dictionaries/lists/strings
        # Passing each value as a list with one string item
        values = df[col].cast(pl.String).to_list()
    
        hasher = FeatureHasher(
            n_features = n_buckets,
            input_type="string",
            alternate_sign=False # simpler interpratation with signs always positive
        )

        hashed = hasher.transform([[v] for v in values])
        hashed_parts.append(hashed)
        print(f" {col}: {n_buckets} buckets, shape: {hashed.shape}")

    # Hash - each value goes to its bucket (1 in that bucket, 0 elsewhere)
    return hstack(hashed_parts).tocsr() # stack all hashed columns horizontally (into one matrix)

# Hashing selected columns for train
print("Hashing train data...")
X_train_hashed = hash_columns(X_train, HASH_BUCKETS)
print(f"\nCombined train hashed shape: {X_train_hashed.shape}")

Hashing train data...
 C14: 1000 buckets, shape: (32377421, 1000)
 C17: 500 buckets, shape: (32377421, 500)
 C19: 100 buckets, shape: (32377421, 100)
 C20: 200 buckets, shape: (32377421, 200)
 C21: 100 buckets, shape: (32377421, 100)
 app_domain: 500 buckets, shape: (32377421, 500)
 app_id: 2000 buckets, shape: (32377421, 2000)
 site_id: 2000 buckets, shape: (32377421, 2000)
 site_domain: 2000 buckets, shape: (32377421, 2000)
 device_model: 2000 buckets, shape: (32377421, 2000)
 device_id: 5000 buckets, shape: (32377421, 5000)
 device_ip: 5000 buckets, shape: (32377421, 5000)

Combined train hashed shape: (32377421, 20400)


In [9]:
# Hash the same columns for val
print("Hashing val data...")
X_val_hashed = hash_columns(X_val, HASH_BUCKETS)
print(f"\nCombined val hashed shape: {X_val_hashed.shape}")

Hashing val data...
 C14: 1000 buckets, shape: (8051546, 1000)
 C17: 500 buckets, shape: (8051546, 500)
 C19: 100 buckets, shape: (8051546, 100)
 C20: 200 buckets, shape: (8051546, 200)
 C21: 100 buckets, shape: (8051546, 100)
 app_domain: 500 buckets, shape: (8051546, 500)
 app_id: 2000 buckets, shape: (8051546, 2000)
 site_id: 2000 buckets, shape: (8051546, 2000)
 site_domain: 2000 buckets, shape: (8051546, 2000)
 device_model: 2000 buckets, shape: (8051546, 2000)
 device_id: 5000 buckets, shape: (8051546, 5000)
 device_ip: 5000 buckets, shape: (8051546, 5000)

Combined val hashed shape: (8051546, 20400)


In [10]:
import gc

# Force Python to release memory (aka garbage collection)
gc.collect()

print("Cleanup done.")

# Check current memort usage
import psutil
mem = psutil.virtual_memory()
print(f"Available memory: {mem.available / 1e9:.2f} GB")
print(f"Used memory:      {mem.used / 1e9:.2f} GB")
print(f"Total memory:     {mem.total / 1e9:.2f} GB")

Cleanup done.
Available memory: 4.36 GB
Used memory:      12.21 GB
Total memory:     16.57 GB


### One-hot encoding

In [11]:
# One-hot encoding for categorical columns (low cardinality)

ohe = OneHotEncoder(
    handle_unknown = "ignore", # val may have values not seen in train
    sparse_output = True, # return sparse matrix (memory efficient)
    drop = "first", # drop first category per column (avoiding multicollinearity)
    dtype = np.float32 # saving memory (default float64)
)

# Fit encoder (to learn all categories) then transform
print("Fitting (OneHotEncoder) and transforming train...")
X_train_ohe = ohe.fit_transform(X_train.select(ONEHOT_COLS).to_pandas())
print(f"Train one-hot shape: {X_train_ohe.shape}")

# Transform val (using already fitted list of values the encoder learned from train data for each column)
print("\nTransforming val...")
X_val_ohe = ohe.transform(X_val.select(ONEHOT_COLS).to_pandas())
print(f"Val one-hot shape: {X_val_ohe.shape}")

# Show category counts per column
print("\nCategories per column:")
for i in range(len(ONEHOT_COLS)):
    col = ONEHOT_COLS[i]
    cats = ohe.categories_[i]
    print(f"  {col:20s}: {len(cats):4d} unique ({len(cats)-1} after drop_first)")

Fitting (OneHotEncoder) and transforming train...
Train one-hot shape: (32377421, 128)

Transforming val...
Val one-hot shape: (8051546, 128)

Categories per column:
  C1                  :    7 unique (6 after drop_first)
  banner_pos          :    7 unique (6 after drop_first)
  device_type         :    5 unique (4 after drop_first)
  device_conn_type    :    4 unique (3 after drop_first)
  C15                 :    8 unique (7 after drop_first)
  C16                 :    9 unique (8 after drop_first)
  C18                 :    4 unique (3 after drop_first)
  app_category        :   36 unique (35 after drop_first)
  site_category       :   26 unique (25 after drop_first)
  hour_of_day         :   24 unique (23 after drop_first)
  day_of_week         :    7 unique (6 after drop_first)
  is_weekend          :    2 unique (1 after drop_first)
  is_app              :    2 unique (1 after drop_first)


### Combining all features

In [12]:
# Combine hashed and one-hot encoded features into single sparse matrix
X_train_final = hstack([X_train_ohe, X_train_hashed]).tocsr()
X_val_final = hstack([X_val_ohe, X_val_hashed]).tocsr()

print(f"Final X_train shape: {X_train_final.shape}")
print(f"Final X_val shape: {X_val_final.shape}")

# Verify y arrays match
print(f"\ny_train shape: {y_train.shape}")
print(f"\ny_val shape: {y_val.shape}")

# Sanity check: number of rows should match between X and y
assert X_train_final.shape[0] == len(y_train), "Train rows mismatch!"
assert X_val_final.shape[0] == len(y_val), "Val rows mismatch!"
assert X_train_final.shape[1] == X_val_final.shape[1], "Column count mismatch!"

print("\nAll dimensions match.")

Final X_train shape: (32377421, 20528)
Final X_val shape: (8051546, 20528)

y_train shape: (32377421,)

y_val shape: (8051546,)

All dimensions match.


## Logistic regression

In [ ]:
import time

# Logistic Regression with saga solver 
model = LogisticRegression(
    solver = "saga", # good for large sparse data
    penalty = "l2", # preventing overfitting, good for corelated features
    C = 1.0, # default, intermediate between weak and strong regularization (fitting to data)
    max_iter = 50, # gradient descent may not converge if number of itarations too small
    tol = 1e-4, # converge tolerance
    n_jobs = 1, # single-threaded avoids threading overhead (-1 use all CPU cores)
    random_state = 42, # reproducibility
    verbose = 1 # show progress
)

print("Training LogisticRegression...")
print(f"Train shape: {X_train_final.shape}, features: {X_train_final.shape[1]:,}")

start_time = time.time()
model.fit(X_train_final, y_train)
elapsed = time.time() - start_time

print(f"\nTraining completed in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")

Training LogisticRegression...
Train shape: (32377421, 20528), features: 20,528


In [14]:
# Try on 1M sample first
import numpy as np

# Random sample of 1M rows for quick test
np.random.seed(42)
sample_size = 1_000_000
sample_idx = np.random.choice(X_train_final.shape[0], size=sample_size, replace=False)

X_train_sample = X_train_final[sample_idx]
y_train_sample = y_train[sample_idx]

print(f"Sample shape: {X_train_sample.shape}")

# Simpler model - single-threaded, fewer iterations
model = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=1.0,
    max_iter=20,
    tol=1e-3,          # looser tolerance for speed
    n_jobs=1,          # single-threaded - avoid potential threading issues
    random_state=42,
    verbose=1,
)

import time
start = time.time()
model.fit(X_train_sample, y_train_sample)
elapsed = time.time() - start
print(f"\nTraining time on 1M sample: {elapsed:.1f}s")

Sample shape: (1000000, 20528)
max_iter reached after 34 seconds

Training time on 1M sample: 34.3s


C:\Users\jagod\anaconda3\envs\ctr-prediction\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   34.1s finished
